# Python Inheritance — Interview Preparation

Covers: single/multiple inheritance, MRO, super(), method overriding, isinstance/issubclass, mixins, and interview Q&A.

## 1. Single Inheritance

In [ ]:
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound

    def speak(self):
        return f'{self.name} says {self.sound}'

    def __repr__(self):
        return f'{type(self).__name__}(name={self.name!r})'


class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, 'Woof')  # call parent __init__
        self.breed = breed

    def speak(self):  # method overriding
        return f'{self.name} barks: Woof woof!'

    def fetch(self):
        return f'{self.name} fetches the ball!'


class Cat(Animal):
    def __init__(self, name):
        super().__init__(name, 'Meow')

    def speak(self):
        parent_result = super().speak()  # call parent method
        return f'{parent_result} (softly)'


dog = Dog('Rex', 'German Shepherd')
cat = Cat('Whiskers')

print(dog.speak())   # Dog's override
print(cat.speak())   # Cat's override + parent
print(dog.fetch())   # Dog-specific method

# Polymorphism
animals = [Dog('Buddy', 'Lab'), Cat('Luna'), Dog('Max', 'Poodle')]
for a in animals:
    print(a.speak())

## 2. MRO — Method Resolution Order

In [ ]:
# Diamond problem — C3 linearization
#     A
#    / \
#   B   C
#    \ /
#     D

class A:
    def method(self):
        return 'A'

class B(A):
    def method(self):
        return 'B'

class C(A):
    def method(self):
        return 'C'

class D(B, C):
    pass

print('D.mro():', [cls.__name__ for cls in D.mro()])
# [D, B, C, A, object]

print('D().method():', D().method())  # 'B' — B comes first in MRO

# A is visited only ONCE (diamond problem solved)
print('A appears once:', D.mro().count(A))

In [ ]:
# Cooperative multiple inheritance with super()
class A:
    def method(self):
        print('A.method')
        # super().method() would call object.method — not defined, so we stop

class B(A):
    def method(self):
        print('B.method')
        super().method()  # calls next in MRO (A)

class C(A):
    def method(self):
        print('C.method')
        super().method()  # calls next in MRO (A)

class D(B, C):
    def method(self):
        print('D.method')
        super().method()  # calls next in MRO (B)

print('MRO:', [cls.__name__ for cls in D.mro()])
print('Calling D().method():')
D().method()
# D → B → C → A  (each super() goes to next in MRO)

## 3. Multiple Inheritance and Mixins

In [ ]:
import json

class JSONMixin:
    """Mixin: adds JSON serialization"""
    def to_json(self):
        return json.dumps(self.__dict__, default=str)

    @classmethod
    def from_json(cls, json_str):
        data = json.loads(json_str)
        obj = cls.__new__(cls)
        obj.__dict__.update(data)
        return obj


class ValidateMixin:
    """Mixin: adds validation"""
    def validate(self):
        for field, validator in getattr(self, '_validators', {}).items():
            value = getattr(self, field, None)
            if not validator(value):
                raise ValueError(f'Invalid {field}: {value}')
        return True


class User(JSONMixin, ValidateMixin):
    _validators = {
        'age': lambda x: x is not None and x >= 0,
        'email': lambda x: x and '@' in x,
    }

    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email


user = User('Alice', 30, 'alice@example.com')
print('JSON:', user.to_json())
print('Valid:', user.validate())

# Restore from JSON
user2 = User.from_json(user.to_json())
print('Restored:', user2.name, user2.age)

## 4. isinstance and issubclass

In [ ]:
class Animal: pass
class Dog(Animal): pass
class Cat(Animal): pass
class GoldenRetriever(Dog): pass

golden = GoldenRetriever()

# isinstance — checks inheritance chain
print('isinstance(golden, GoldenRetriever):', isinstance(golden, GoldenRetriever))  # True
print('isinstance(golden, Dog):', isinstance(golden, Dog))    # True
print('isinstance(golden, Animal):', isinstance(golden, Animal))  # True
print('isinstance(golden, Cat):', isinstance(golden, Cat))    # False

# isinstance with tuple — OR check
print('isinstance(golden, (Dog, Cat)):', isinstance(golden, (Dog, Cat)))  # True

# issubclass — class hierarchy
print('issubclass(GoldenRetriever, Dog):', issubclass(GoldenRetriever, Dog))    # True
print('issubclass(GoldenRetriever, Animal):', issubclass(GoldenRetriever, Animal))  # True
print('issubclass(Dog, Dog):', issubclass(Dog, Dog))  # True — class is subclass of itself

# type() — exact type only
print('type(golden) is GoldenRetriever:', type(golden) is GoldenRetriever)  # True
print('type(golden) is Dog:', type(golden) is Dog)  # False

## 5. Interview Practice

In [ ]:
# Q: Why use super() instead of calling parent class directly?
class Base:
    def __init__(self, x):
        self.x = x
        print(f'Base.__init__({x})')

class Mixin:
    def __init__(self, **kwargs):
        print('Mixin.__init__')
        super().__init__(**kwargs)

# With super() — cooperative, follows MRO
class Child(Mixin, Base):
    def __init__(self, x):
        print('Child.__init__')
        super().__init__(x=x)  # follows MRO: Child → Mixin → Base

print('MRO:', [c.__name__ for c in Child.mro()])
c = Child(42)

In [ ]:
# Q: Implement a Shape hierarchy with polymorphism
import math

class Shape:
    def area(self):
        raise NotImplementedError

    def perimeter(self):
        raise NotImplementedError

    def describe(self):
        return f'{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}'


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)


shapes = [Circle(5), Rectangle(4, 6), Circle(3)]
for s in shapes:
    print(s.describe())

total_area = sum(s.area() for s in shapes)
print(f'Total area: {total_area:.2f}')